# marv-hyena round 2: fixing round 1's experiment design

Round 1 ([RESEARCH_LOG.md](https://github.com/thebnbrkr/marv-hyena/blob/main/RESEARCH_LOG.md)) found two things
that changed the plan:

1. **Block 30 (the last LI block) got all of the direct credit** for the model's predictions. The hypothesis is that
   its output is far larger than every other block's, so the output layer effectively reads only block 30.
2. **Switching off a whole operator type broke the model.** SE, MR and LI families all pushed it to chance-level
   guessing, so those rows measured "broken model", not "what this part does".

Round 2:
- **R2.1** checks the block-30 hypothesis directly (the output size of every block).
- Every ablation now carries a **health check** (is the model still working?).
- Families are ablated **without the bottleneck block**, and also **one layer at a time**.
- The motif search gets a **composition control** and a **per-position importance** map.
- BRCA1 patching is done **at the mutation site only**, on variants with a real effect, plus a small
  **harmful-vs-harmless scoring check** across 40 variants.

**Runtime:** A100 + High-RAM, as before.

## 0. Setup

In [ ]:
# GPU check (no torch import yet: the install below may change the torch version)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# OPTIONAL: keep the ~15 GB Evo 2 weights on Google Drive so the next session skips the download.
# This must run BEFORE anything imports huggingface_hub / evo2.
USE_DRIVE = False

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ.get('HF_HOME', '(default ~/.cache/huggingface)'))

In [ ]:
# Install Evo 2 + marv-hyena (~2-5 min). NO flash-attn: pip usually finds no prebuilt wheel for Colab's torch
# and compiles for hours. marv-hyena instead runs Evo 2's attention through PyTorch's built-in fused kernel
# (scaled_dot_product_attention), which is also fast on an A100 -- see marv_hyena/noflash.py.
# Colab runs Python 3.13, but every current evo2 release declares Python < 3.13, so a plain `pip install evo2`
# silently falls back to the old, incompatible 0.3.0. evo2 is pure Python, so force the current release.
# Colab also ships an empty `transformer-engine` package whose import crashes evo2; remove it (7B needs no TE).
!pip uninstall -y -q transformer-engine transformer_engine
!pip install -q --ignore-requires-python evo2==0.6.0
!rm -rf marv-hyena && git clone -q https://github.com/thebnbrkr/marv-hyena.git
!pip install -q -r marv-hyena/requirements.txt openpyxl

import sys
sys.path.insert(0, '/content/marv-hyena')

In [ ]:
# Must run before anything imports evo2/vortex: lets Vortex import without flash-attn and turns flash attention off.
from marv_hyena import noflash
noflash.prepare()

import torch, vortex, evo2
from importlib.metadata import version
print('evo2', version('evo2'), '| vtx', version('vtx'))
assert version('evo2') >= '0.6.0', 'old evo2 installed: Runtime -> Disconnect and delete runtime, then rerun from the top'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| flash-attn package installed:', noflash.flash_attn_available())
print('GPU', torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'needs an Ampere+ GPU with bf16 (A100 / L4 / H100)'

In [ ]:
# marv-hyena's own tests (a tiny CPU model that mirrors Vortex). Should say "29 passed".
!cd marv-hyena && python -m pytest -q

In [ ]:
# Data: the annotated E. coli K-12 genome from the evo2 repo
import os, urllib.request
os.makedirs('data', exist_ok=True)
EVO2_RAW = 'https://raw.githubusercontent.com/ArcInstitute/evo2/main/notebooks'
GENOME = 'data/NC_000913.gb'
if not os.path.exists(GENOME):
    urllib.request.urlretrieve(f'{EVO2_RAW}/sparse_autoencoder/NC_000913.gb', GENOME)

import marv_hyena as mh
from marv_hyena.probes import load_sequence
genome = load_sequence(GENOME)
print(f'E. coli genome: {len(genome):,} letters')

In [ ]:
# Load Evo 2 7B (downloads ~15 GB the first time)
import time
MODEL_NAME = 'evo2_7b'
t0 = time.time()
hm = mh.HyenaModel.load(MODEL_NAME)
print(f'loaded in {time.time()-t0:.0f}s')
print(hm.describe())
RESULTS = {}

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from marv_hyena import diagnostics, motifs
from marv_hyena.checks import run_smoke_checks
from marv_hyena.experiments import make_conditions, copy_test, codon_test, context_test, periodicity, summarize_copy

seq = genome[100_000:104_096]
assert run_smoke_checks(hm, seq), 'smoke checks failed: stop here'
HEALTH_SEQ = genome[300_000:304_096]          # ordinary genome used to check the model still works
print('baseline health:', diagnostics.health(hm, HEALTH_SEQ))
RESULTS = {}

## R2.1 Is block 30 an output bottleneck?

Measure the size (L2 norm) of every block's residual write, and what share of the final residual it makes up, at
positions in three different stretches of genome. If block 30's mixer is most of the final residual everywhere, the
hypothesis holds.

In [ ]:
norm_rows = []
for start in (100_000, 1_000_000, 3_000_000):
    s = genome[start:start + 4096]
    rows = diagnostics.write_norms(hm, s, [1000, 2000, 4095])
    print(f'--- genome[{start}:+4096]')
    diagnostics.show_write_norms(rows, k=6)
    norm_rows.append(rows)

BOTTLENECK = diagnostics.find_bottlenecks(hm, seq, [1000, 2000, 4095])
print('\nbottleneck blocks (write >= 50% of final residual norm):', BOTTLENECK)
RESULTS['write_norms'] = [[{'block': r.block, 'kind': r.kind, 'part': r.part, 'norm': r.norm, 'share': r.share}
                           for r in rows] for rows in norm_rows]
RESULTS['bottleneck'] = BOTTLENECK

rows = norm_rows[0]
labels = ['emb'] + [f'{r.block}{r.part[0]}' for r in rows[1:]]
cols = ['grey'] + [{'se': 'C0', 'mr': 'C1', 'li': 'C2', 'attn': 'C3'}[r.kind] for r in rows[1:]]
plt.figure(figsize=(14, 3))
plt.bar(range(len(rows)), [r.norm for r in rows], color=cols)
plt.yscale('log'); plt.xticks(range(len(rows)), labels, rotation=90, fontsize=7)
plt.ylabel('write norm (log)'); plt.title('size of each write (m = mixer, m/l = mlp); SE blue MR orange LI green attn red')
plt.show()

## R2.2 One layer at a time: what breaks?

For each of the 32 blocks, switch off **only that block's mixer** (replaced with its average output) and measure:
- **health**: next-letter accuracy on ordinary genome (is the model still working?);
- **codon rhythm**: accuracy at codon positions 1–2 minus position 3;
- **copying**: second-copy accuracy with gaps of 1,000 and 10,000 letters.

This is the table that tells us which single layers are load-bearing for which job.

In [ ]:
track = mh.genbank_track(GENOME, 200_000, 208_192)
single = make_conditions(hm, families=(), single_blocks=range(hm.n_blocks))
codon_rows = codon_test(hm, track, conditions=single)
per = periodicity(codon_rows)
copy_rows = copy_test(hm, genome, gaps=(1000, 10000), insert_len=200, seeds=1, conditions=single, verbose=False)

layer_table = []
for name in single:
    h = next(r for r in codon_rows if r['condition'] == name)
    c = {g: np.mean([r['second_acc'] for r in copy_rows if r['condition'] == name and r['gap'] == g]) for g in (1000, 10000)}
    layer_table.append({'condition': name, 'health_acc': h['health_acc'], 'broken': h['broken'],
                        'codon_rhythm': per[name], 'copy@1k': c[1000], 'copy@10k': c[10000]})
df_layers = pd.DataFrame(layer_table).set_index('condition')
RESULTS['single_layer'] = layer_table
df_layers.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 7), sharey=True)
for ax, col in zip(axes, ['health_acc', 'codon_rhythm', 'copy@1k', 'copy@10k']):
    v = df_layers[col].values
    ax.barh(range(len(v)), v, color=['C3' if '(attn)' in n else 'C2' if '(li)' in n else 'C1' if '(mr)' in n else 'C0'
                                     for n in df_layers.index])
    ax.axvline(df_layers.loc['none', col], color='k', ls='--', lw=0.8)
    ax.set_title(col); ax.invert_yaxis()
axes[0].set_yticks(range(len(df_layers))); axes[0].set_yticklabels(df_layers.index, fontsize=7)
plt.suptitle('single-mixer ablations (dashed = unablated model)'); plt.tight_layout(); plt.show()

## R2.3 Whole families again, this time without the bottleneck

The same family ablations as round 1, but block 30 (or whatever R2.1 found) is kept switched on. Rows where
`broken` is True are still uninterpretable. Report them, but don't read meaning into them.

In [ ]:
fam = make_conditions(hm, exclude_blocks=BOTTLENECK)
print('conditions:', list(fam))

copy2 = copy_test(hm, genome, gaps=(100, 1000, 10000), insert_len=200, seeds=2, conditions=fam,
                  health_seq=HEALTH_SEQ, verbose=False)
gaps, conds, a2, a1 = summarize_copy(copy2)
health_by = {r['condition']: (r['health_acc'], r['broken']) for r in copy2}
print(f"{'condition':<22} {'health':>7} {'broken':>7} " + ' '.join(f'copy@{g:>6}' for g in gaps))
for j, c in enumerate(conds):
    print(f'{c:<22} {health_by[c][0]:>7.3f} {str(health_by[c][1]):>7} ' + ' '.join(f'{a2[i, j]:>11.3f}' for i in range(len(gaps))))
RESULTS['copy_families'] = copy2

In [ ]:
codon2 = codon_test(hm, track, conditions=fam)
RESULTS['codon_families'] = codon2
t = pd.DataFrame(codon2).pivot(index='condition', columns='region', values='acc')
t['rhythm'] = pd.Series(periodicity(codon2))
t['health_acc'] = pd.DataFrame(codon2).groupby('condition')['health_acc'].first()
t['broken'] = pd.DataFrame(codon2).groupby('condition')['broken'].first()
t.round(3)

In [ ]:
# P2 on 5 genes instead of 1: does far-away DNA help, and which family carries it?
from Bio import SeqIO
rec = next(SeqIO.parse(GENOME, 'genbank'))
UP = 50_000
cands = [f for f in rec.features if f.type == 'CDS' and f.location.strand == 1
         and int(f.location.start) > UP + 10_000 and len(f.location) > 1200]
genes = [cands[i] for i in np.linspace(0, len(cands) - 1, 5).astype(int)]   # 5 genes spread across the genome
ctx_rows = []
for f in genes:
    s = int(f.location.start)
    rows = context_test(hm, genome[s - UP: s + 1000], target=(UP, UP + 1000), short=500, conditions=fam)
    for r in rows:
        r['gene'] = f.qualifiers.get('gene', ['?'])[0]
    ctx_rows += rows
    print('done', rows[0]['gene'])
RESULTS['context_families'] = ctx_rows
dfc = pd.DataFrame(ctx_rows)
dfc.pivot(index='condition', columns='gene', values='context_benefit').round(4)

## R2.4 First-layer motifs, with controls

Block 0 sees the last 9 letters. We run all 262,144 possible 9-letter inputs through it, then:
- **per-position importance**: for each channel, how much each of the 9 positions affects its output. This is exact,
  from the complete enumeration. It tests the round-1 impression that channels mostly care about the last 5–6 letters.
- **motif counts with a composition control**: a channel only counts as, say, a "TAA channel" if TAA appears in its
  top inputs ≥ 3× more often than chance for a channel with the same A/C/G/T mix. This stops AT-rich channels from
  being counted as TAA channels.

In [ ]:
md_ = motifs.enumerate_block0(hm)
imp = md_.position_importance                     # (9, 4096), columns sum to 1
prof = imp.mean(1)
late = imp[-6:].sum(0)                            # share of each channel's importance in the last 6 positions
print('mean importance by position (0 = 8 letters back, 8 = the current letter):', np.round(prof, 3))
print(f'channels with >= 80% of importance in the last 6 positions: {(late >= 0.8).mean():.1%}')
print(f'channels with >= 50% in the current letter alone: {(imp[-1] >= 0.5).mean():.1%}')
RESULTS['motif_position_profile'] = prof.tolist()

plt.figure(figsize=(6, 3))
plt.bar(range(-8, 1), prof); plt.xlabel('position (0 = current letter)'); plt.ylabel('mean importance')
plt.title('block 0: how much each input position matters'); plt.show()

motif_summary = []
for name, m in [('start ATG', 'ATG'), ('stop TAA', 'TAA'), ('stop TAG', 'TAG'), ('stop TGA', 'TGA'),
                ('Shine-Dalgarno', 'AGGAG'), ('control CCC', 'CCC'), ('control GCG', 'GCG')]:
    raw, ctrl = motifs.motif_channels(md_, m)
    motif_summary.append({'motif': name, 'raw_channels': len(raw), 'controlled_channels': len(ctrl),
                          'examples': [h.channel for h in sorted(ctrl, key=lambda h: -h.enrichment)[:5]]})
RESULTS['motif_counts'] = motif_summary
pd.DataFrame(motif_summary)

In [ ]:
# Look at the best controlled channel for each real motif: its top inputs and letter preferences per position
for row in motif_summary[:5]:
    if not row['examples']:
        print(f"{row['motif']}: no channel passes the control"); continue
    c = row['examples'][0]
    pwm = md_.pwm(c)
    print(f"\n{row['motif']}: channel {c}; top inputs {md_.top_kmers[c][:5]}")
    print('pos  ' + '  '.join(f'{p:>2}' for p in range(-8, 1)))
    for j, b in enumerate('ACGT'):
        print(f'  {b}  ' + '  '.join(f'{pwm[i, j]:.1f}'[1:] if pwm[i, j] < 1 else '1.' for i in range(9)))
    print('importance ' + ' '.join(f'{x:.2f}' for x in md_.position_importance[:, c]))

## R2.5 BRCA1: does Evo 2 separate harmful from harmless, and where does the mutation's signal enter?

1. Score 20 loss-of-function (LOF) and 20 functional (FUNC) variants, and check whether LOF variants score lower
   (AUROC: 0.5 = no better than chance, 1.0 = perfect separation).
2. For the 3 LOF variants with the largest effect on the letters that follow, patch each component **at the mutation
   site only** (the round-1 all-position patching mostly measured the block-30 bottleneck).

In [ ]:
import gzip, os, urllib.request
from sklearn.metrics import roc_auc_score
for f in ['GRCh37.p13_chr17.fna.gz', '41586_2018_461_MOESM3_ESM.xlsx']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'{EVO2_RAW}/brca1/{f}', f'data/{f}')
with gzip.open('data/GRCh37.p13_chr17.fna.gz', 'rt') as h:
    chr17 = str(next(SeqIO.parse(h, 'fasta')).seq).upper()
brca1 = pd.read_excel('data/41586_2018_461_MOESM3_ESM.xlsx', header=2)[
    ['chromosome', 'position (hg19)', 'reference', 'alt', 'function.score.mean', 'func.class']]
sample = pd.concat([brca1[brca1['func.class'] == 'LOF'].sample(20, random_state=0),
                    brca1[brca1['func.class'] == 'FUNC'].sample(20, random_state=0)])
var_rows = []
for _, row in sample.iterrows():
    v = mh.make_variant(chr17, int(row['position (hg19)']) - 1, row['reference'], row['alt'], window=8192)
    var_rows.append({'pos': int(row['position (hg19)']), 'ref': row['reference'], 'alt': row['alt'],
                     'class': row['func.class'], 'lab_score': float(row['function.score.mean']),
                     'delta_logp': mh.delta_logp(hm, v), 'downstream_effect': mh.variants.downstream_effect(hm, v)})
dv = pd.DataFrame(var_rows)
auc = roc_auc_score((dv['class'] == 'LOF').astype(int), -dv['delta_logp'])
print(f'AUROC (LOF vs FUNC, lower delta_logp = more harmful): {auc:.3f}   (n=40; the Evo 2 paper uses thousands)')
print(dv.groupby('class')[['delta_logp', 'downstream_effect']].mean())
RESULTS['brca1_scores'] = var_rows
RESULTS['brca1_auroc_n40'] = auc

In [ ]:
top = dv[dv['class'] == 'LOF'].reindex(dv[dv['class'] == 'LOF']['downstream_effect'].abs().sort_values(ascending=False).index).head(3)
site_rows = []
for _, row in top.iterrows():
    if abs(row['downstream_effect']) < 1.0:
        print(f"skip {row['pos']}: effect {row['downstream_effect']:+.2f} too small to split"); continue
    v = mh.make_variant(chr17, row['pos'] - 1, row['ref'], row['alt'], window=8192)
    sweep = mh.explain_variant(hm, v, span=200, at='variant')
    print(f"\n===== chr17:{row['pos']} {row['ref']}>{row['alt']}  downstream effect {row['downstream_effect']:+.2f}")
    sweep.show(k=10)
    for r in sweep.results:
        name = r.component if isinstance(r.component, str) else f'L{r.component[0]} {r.kind} {r.component[1]}'
        site_rows.append({'variant': row['pos'], 'component': name, 'fraction': r.fraction})
RESULTS['brca1_site_patching'] = site_rows
if site_rows:
    ds = pd.DataFrame(site_rows).pivot(index='component', columns='variant', values='fraction')
    display(ds.loc[[i for i in ds.index if i.startswith('all')]].round(3))

## Save

Download `results_round2.json` (or it goes to Drive if you mounted it), then send it back together with the executed
notebook.

In [ ]:
out = '/content/drive/MyDrive/marv_hyena_results_round2.json' if USE_DRIVE else 'results_round2.json'
with open(out, 'w') as f:
    json.dump(RESULTS, f, indent=1, default=float)
print('wrote', out)